In [ ]:
df_silver = spark.read.table('silver.yellow_taxi_tripdata')
display(df_silver)

In [ ]:
from pyspark.sql import functions as F

**read dataframe**

In [8]:
fact_taxi = df_silver.select('VendorID','pickup_datetime','dropoff_datetime','passenger_count','trip_distance','RatecodeID','PULocationID','DOLocationID','fare_amount','tip_amount','total_amount')\
                    .dropDuplicates(['VendorID','pickup_datetime','dropoff_datetime','PULocationID','DOLocationID'])
dim_location = spark.read.option('header','true').csv('abfss://986a8382-8e44-4e7f-9f46-bf733c7d5da7@onelake.dfs.fabric.microsoft.com/3c5d469b-719f-4960-a8d4-be0dd4a47cc6/Files/csv/taxi_zone_lookup.csv')
dim_location = dim_location.withColumn('LocationID',F.col('LocationID').cast('int'))

StatementMeta(, 92b68b6d-5bbc-4b48-b93f-9b409185f226, 10, Finished, Available, Finished, False)

**write to gold tables**

In [10]:
fact_taxi.write.mode('overwrite').format('delta').saveAsTable('gold.fact_yellowtaxi')
dim_location.write.mode('overwrite').format('delta').option('overwriteSchema','true').saveAsTable('gold.dim_location')

StatementMeta(, 92b68b6d-5bbc-4b48-b93f-9b409185f226, 12, Finished, Available, Finished, False)

**optimize**

In [ ]:
%%sql
OPTIMIZE gold.fact_yellowtaxi

**vacuum**

In [ ]:
# 1. Cho phép Vacuum ngay lập tức (mặc định nó bắt đợi 7 ngày)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# 2. Chạy dọn rác cho bảng Gold
spark.sql("VACUUM gold.fact_yellowtaxi RETAIN 0 HOURS")

**tạo bảng dim_date**

In [14]:
start_date = '2025-01-01'
end_date  = '2025-12-31'
df_date = spark.sql(f"select explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) as date")
dim_date = df_date.select(
                    F.col('date').alias('date_key'),
                    F.year('date').alias('year'),
                    F.month('date').alias('month'),
                    F.dayofmonth('date').alias('day'),
                    F.date_format('date','MMM').alias('month_name'),
                    F.date_format('date','EEEE').alias('day_name'),
                    F.quarter('date').alias('quarter'),
                    F.weekofyear('date').alias('week_of_year')
                        )
dim_date.write.mode("overwrite").format("delta").saveAsTable("gold.dim_date")

StatementMeta(, 92b68b6d-5bbc-4b48-b93f-9b409185f226, 16, Finished, Available, Finished, False)